# Bài 18: Logging
## Ghi lại những gì đang xảy ra trong hệ thống

**Mục tiêu:**
- Hiểu tại sao `print()` không đủ trong production
- Dùng module `logging` của Python
- Thêm logging vào `app.py`

---
## Phần 1: Lý thuyết

### Vấn đề với `print()`

```python
print("Đang xử lý NVDA...")   # không biết lúc nào, từ đâu
print("Xong!")                 # không có timestamp, không có cấp độ
```

Trong production:
- Không biết log xảy ra lúc mấy giờ
- Không phân biệt được thông tin thường và lỗi nghiêm trọng
- Không thể tắt log debug mà không xóa code
- Không lưu ra file được

---

### Log levels — cấp độ quan trọng

| Level | Dùng khi nào |
|-------|-------------|
| `DEBUG` | Chi tiết kỹ thuật — chỉ cần khi debug |
| `INFO` | Luồng chạy bình thường — "Đang xử lý NVDA" |
| `WARNING` | Bất thường nhưng không crash — "Không tìm thấy báo cáo" |
| `ERROR` | Lỗi thực sự — exception, API fail |
| `CRITICAL` | Hệ thống sắp crash |

Khi bạn set level `INFO` → chỉ thấy INFO, WARNING, ERROR, CRITICAL (DEBUG bị ẩn).

---

### Cấu trúc logging

```
Logger  →  Handler  →  Formatter
(ai log)   (đi đâu)    (hiện thế nào)
```

- **Logger**: đối tượng bạn gọi `.info()`, `.error()` — đặt tên theo module
- **Handler**: `StreamHandler` (terminal) hoặc `FileHandler` (file)
- **Formatter**: định dạng dòng log — timestamp, level, tên, nội dung

---

### Setup cơ bản

```python
import logging

def setup_logger(name: str, log_file: str = None) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)  # logger nhận tất cả

    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S"
    )

    # Handler 1: in ra terminal
    sh = logging.StreamHandler()
    sh.setLevel(logging.INFO)   # terminal chỉ hiện INFO+
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    # Handler 2: ghi ra file (tùy chọn)
    if log_file:
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setLevel(logging.DEBUG)  # file ghi cả DEBUG
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    return logger
```

Kết quả log trông như thế này:
```
12:05:03 | INFO     | stock_app | Bắt đầu xử lý NVDA
12:05:04 | DEBUG    | stock_app | Tìm thấy 3 chunks từ ChromaDB
12:05:06 | WARNING  | stock_app | Không có báo cáo cho AMD
12:05:08 | ERROR    | stock_app | API call thất bại: 503
```

---
## Phần 2: Ví dụ

Demo logging trong một hàm đơn giản, quan sát sự khác nhau giữa các level.

In [1]:
import logging
import time

# ── Setup logger ──────────────────────────────────────────────────────────────
def setup_logger(name: str, log_file: str = None) -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:          # tránh add handler 2 lần khi re-run cell
        return logger
    logger.setLevel(logging.DEBUG)

    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S"
    )

    sh = logging.StreamHandler()
    sh.setLevel(logging.DEBUG)
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    if log_file:
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    return logger


# ── Demo ──────────────────────────────────────────────────────────────────────
logger = setup_logger("demo", log_file="demo.log")

def process_symbol(symbol: str):
    logger.info(f"Bắt đầu xử lý {symbol}")
    start = time.time()

    try:
        logger.debug(f"{symbol}: gọi yfinance...")
        time.sleep(0.1)   # giả lập network call

        # giả lập: AMD không có báo cáo
        if symbol == "AMD":
            logger.warning(f"{symbol}: không tìm thấy báo cáo trong ChromaDB")

        elapsed = time.time() - start
        logger.info(f"{symbol}: hoàn thành sau {elapsed:.2f}s")

    except Exception as e:
        logger.error(f"{symbol}: lỗi — {e}")


for sym in ["NVDA", "AMD", "AAPL"]:
    process_symbol(sym)

09:51:17 | INFO     | demo | Bắt đầu xử lý NVDA
09:51:17 | DEBUG    | demo | NVDA: gọi yfinance...
09:51:17 | INFO     | demo | NVDA: hoàn thành sau 0.10s
09:51:17 | INFO     | demo | Bắt đầu xử lý AMD
09:51:17 | DEBUG    | demo | AMD: gọi yfinance...
09:51:17 | WARNING  | demo | AMD: không tìm thấy báo cáo trong ChromaDB
09:51:17 | INFO     | demo | AMD: hoàn thành sau 0.10s
09:51:17 | INFO     | demo | Bắt đầu xử lý AAPL
09:51:17 | DEBUG    | demo | AAPL: gọi yfinance...
09:51:17 | INFO     | demo | AAPL: hoàn thành sau 0.10s


In [2]:
# Đọc lại file log để thấy DEBUG cũng được lưu
with open("demo.log", encoding="utf-8") as f:
    print(f.read())

09:51:17 | INFO     | demo | Bắt đầu xử lý NVDA
09:51:17 | DEBUG    | demo | NVDA: gọi yfinance...
09:51:17 | INFO     | demo | NVDA: hoàn thành sau 0.10s
09:51:17 | INFO     | demo | Bắt đầu xử lý AMD
09:51:17 | DEBUG    | demo | AMD: gọi yfinance...
09:51:17 | WARNING  | demo | AMD: không tìm thấy báo cáo trong ChromaDB
09:51:17 | INFO     | demo | AMD: hoàn thành sau 0.10s
09:51:17 | INFO     | demo | Bắt đầu xử lý AAPL
09:51:17 | DEBUG    | demo | AAPL: gọi yfinance...
09:51:17 | INFO     | demo | AAPL: hoàn thành sau 0.10s



---
## Phần 3: Bài tập

Thêm logging vào `phase4/app.py`. Sau khi hoàn thành, khi chạy Streamlit bạn sẽ thấy log xuất hiện trong terminal theo dạng:

```
12:05:01 | INFO     | stock_app | [data_collection] Bắt đầu thu thập cho 2 symbols: ['NVDA', 'AMD']
12:05:02 | INFO     | stock_app | [collect] NVDA: bắt đầu
12:05:03 | DEBUG    | stock_app | [collect] NVDA: tìm thấy 3 chunks từ ChromaDB
12:05:04 | INFO     | stock_app | [collect] NVDA: xong sau 1.23s
12:05:05 | WARNING  | stock_app | [collect] AMD: không có báo cáo liên quan
12:05:07 | INFO     | stock_app | [analysis] Bắt đầu phân tích, context dài 1842 ký tự
12:05:10 | INFO     | stock_app | [analysis] Hoàn thành sau 3.12s
```

---

### Bước 1: Thêm `setup_logger()` vào `app.py`

Copy hàm `setup_logger()` từ Phần 2 vào `app.py`, đặt ngay sau phần `# ── Setup`.

Sau đó khởi tạo logger một lần duy nhất ở module level:
```python
logger = setup_logger("stock_app", log_file="phase4/app.log")
```

---

### Bước 2: Thêm logging vào `collect_for_symbol()`

Hàm này cần log:
- **INFO** khi bắt đầu xử lý symbol
- **DEBUG** về số chunks tìm được từ ChromaDB (`len(report_chunks)`)
- **WARNING** nếu `report_chunks` rỗng
- **INFO** khi hoàn thành kèm thời gian chạy

Gợi ý cách đo thời gian:
```python
import time
start = time.time()
# ... code ...
elapsed = time.time() - start
logger.info(f"... xong sau {elapsed:.2f}s")
```

---

### Bước 3: Thêm logging vào `data_collection_node()`

Log **INFO** ở đầu hàm, ghi rõ bao nhiêu symbols và tên chúng là gì.

---

### Bước 4: Thêm logging vào `analysis_node()`

Log:
- **INFO** khi bắt đầu, ghi độ dài `context` (số ký tự)
- **INFO** khi hoàn thành kèm thời gian

---

### Bước 5: Kiểm tra

Chạy Streamlit và đặt câu hỏi về NVDA và AMD. Terminal phải hiện log đúng format. Sau đó đọc file `phase4/app.log` để xác nhận DEBUG cũng được lưu.